# **Favorita 판매 데이터 탐색**

다음 주 수요를 계획하려면 판매가 **언제, 어디서, 얼마나 달라지는지** 먼저 알아야 합니다. 이 노트북은 매장·상품군의 판매 규모, 0 판매, 요일 반복, 판촉과 공휴일의 단순 차이를 확인합니다.

관찰된 패턴은 기준선과 예측 변수의 근거가 됩니다. 이곳의 집계는 설명용이며 판촉이나 휴일의 인과효과를 뜻하지 않습니다.

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## **Library**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from importlib.util import find_spec
from matplotlib import font_manager

## **Utils**


In [ ]:
# 필요한 원본 열만 읽어 탐색용 표를 만듭니다.
def load_data(raw_path):
    train = pd.read_csv(
        raw_path / "train.csv",
        usecols=["date", "store_nbr", "family", "sales", "onpromotion"],
        parse_dates=["date"],
        dtype={"store_nbr": "int16", "family": "category", "sales": "float32", "onpromotion": "int16"},
    )
    stores = pd.read_csv(raw_path / "stores.csv")
    transactions = pd.read_csv(raw_path / "transactions.csv", parse_dates=["date"])
    return train, stores, transactions

In [ ]:
# 매장과 상품군을 합친 일별 판매량을 계산합니다.
def daily_sales(frame):
    return frame.groupby("date", as_index=False, observed=True)["sales"].sum()

In [ ]:
# 판촉 상품 유무에 따른 판매 분포를 기술적으로 비교합니다.
def promotion_summary(frame):
    return (
        frame.assign(promoted=frame["onpromotion"].gt(0))
        .groupby("promoted", observed=True)["sales"]
        .agg(rows="size", mean="mean", median="median")
    )

## **실행 설정**


In [ ]:
font_path = Path(find_spec("koreanize_matplotlib").origin).parent / "fonts" / "NanumGothic.ttf"
font_manager.fontManager.addfont(font_path)
sns.set_theme(style="whitegrid", font=font_manager.FontProperties(fname=font_path).get_name())
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = REPO
RAW_PATH = PROJECT_ROOT / "data" / "raw"

In [ ]:
train, stores, transactions = load_data(RAW_PATH)
holidays = pd.read_csv(RAW_PATH / "holidays_events.csv", parse_dates=["date"])


## **원본 데이터와 이름표**

- 아래 표는 각 파일의 **첫 5행 예시**입니다. 파일마다 한 행이 뜻하는 단위와 날짜·매장·상품군 열을 먼저 확인합니다.
- `transactions.csv`의 거래 건수는 판매 수량과 다릅니다. 이후 수요 예측의 정답은 `train.csv`의 `sales`입니다.

### **판매 기록 · train.csv**


In [ ]:
train.head()


### **매장 정보 · stores.csv**


In [ ]:
stores.head()


### **거래 건수 · transactions.csv**


In [ ]:
transactions.head()


### **휴일·행사 · holidays_events.csv**


In [ ]:
holidays.head()


## **데이터 규모와 범위 확인**

- 각 원본 파일의 행 수·기간·결측을 확인합니다. 판매 행은 매장×상품군×날짜 단위이므로 전체 판매량을 보려면 합산해야 합니다.
- 이 표는 데이터가 어느 기간과 매장을 설명하는지 알려주며, **예측 정확도나 발주 성과를 뜻하지는 않습니다.**

In [ ]:
train_overview = pd.Series({
    "행 수": len(train),
    "첫 판매일": train["date"].min().date(),
    "마지막 판매일": train["date"].max().date(),
    "매장 수": train["store_nbr"].nunique(),
    "상품군 수": train["family"].nunique(),
    "판매량 결측": train["sales"].isna().sum(),
    "판촉 결측": train["onpromotion"].isna().sum(),
}, name="train.csv")
train_overview


In [ ]:
stores_overview = pd.Series({
    "행 수": len(stores),
    "매장 수": stores["store_nbr"].nunique(),
    "도시 수": stores["city"].nunique(),
    "전체 결측": stores.isna().sum().sum(),
}, name="stores.csv")
stores_overview


In [ ]:
transactions_overview = pd.Series({
    "행 수": len(transactions),
    "첫 거래일": transactions["date"].min().date(),
    "마지막 거래일": transactions["date"].max().date(),
    "매장 수": transactions["store_nbr"].nunique(),
    "거래 건수 결측": transactions["transactions"].isna().sum(),
}, name="transactions.csv")
transactions_overview


In [ ]:
holidays_overview = pd.Series({
    "행 수": len(holidays),
    "첫 행사일": holidays["date"].min().date(),
    "마지막 행사일": holidays["date"].max().date(),
    "행사 유형 수": holidays["type"].nunique(),
    "전체 결측": holidays.isna().sum().sum(),
}, name="holidays_events.csv")
holidays_overview


## **데이터 열을 읽는 법**

예시값은 원본 파일의 실제 행에서 가져왔습니다. 숫자로 저장된 매장 번호와 점포 군집은 크기보다 **구분**이 중요합니다.

### **핵심 판매 데이터: `train.csv`**

| 열 | 의미 | 예시값 | 예측에서의 역할·주의사항 |
|---|---|---|---|
| `date` | 판매일 | `2013-01-01` | 시간순 학습·검증에 사용 |
| `store_nbr` | 매장 번호 | `1` | 매장을 구분하며 숫자의 크기에는 의미가 없음 |
| `family` | 상품군 | `AUTOMOTIVE` | 매장×상품군 시계열을 구분 |
| `sales` | 해당 일의 판매량 | `0.0` | 목표일 판매량은 정답이며, 기준일 이전 값만 입력 가능 |
| `onpromotion` | 판촉 중인 상품 수 | `0` | 미래 계획을 미리 알 수 있을 때만 목표일 입력으로 사용 가능 |

### **매장·거래·휴일 데이터**

| 파일·열 | 의미 | 예시값 | 예측에서의 역할·주의사항 |
|---|---|---|---|
| `stores.store_nbr` | 매장 번호 | `1` | 판매 기록과 결합하는 키 |
| `stores.city` | 도시 | `Quito` | 지역 수요와 휴일을 연결 |
| `stores.state` | 주 | `Pichincha` | 지역 차이를 구분 |
| `stores.type` | 매장 유형 | `D` | 매장 특성의 대리 변수 |
| `stores.cluster` | 기존 유사 매장 번호 | `13` | 수치 크기의 순서로 해석하지 않음 |
| `transactions.transactions` | 하루 거래 건수 | `2111` | 해당 매장의 과거 수요 수준; 목표일 건수는 사용 불가 |
| `holidays_events.type` | 휴일·행사 유형 | `Holiday` | 달력 효과를 나타내는 변수 |
| `holidays_events.locale` | 적용 범위 | `Local` | 전국·지역·도시를 구분 |
| `holidays_events.locale_name` | 적용 지역 | `Manta` | 매장 지역과 연결 |
| `holidays_events.description` | 행사 설명 | `Fundacion de Manta` | 일부 이벤트만 파생 변수로 사용 |
| `holidays_events.transferred` | 휴일 이관 여부 | `False` | 이관된 원래 날짜를 실제 휴일로 세지 않음 |

## **판매량 분포: 보조 진단**

- 가로축은 **매장×상품군×하루**의 판매 수량입니다. 매장 전체 매출이나 상품 한 개의 판매량이 아닙니다.
- 0 판매와 큰 판매량이 함께 있어 평가 지표를 여러 개 써야 함을 보여줍니다. 발주 우선순위는 뒤의 상품군별 판매 규모와 예측 오류에서 판단합니다.

In [ ]:
sample = train["sales"].sample(min(300_000, len(train)), random_state=42).clip(lower=0)
zero_rate = train["sales"].eq(0).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(sample.clip(upper=sample.quantile(0.99)), bins=60, ax=axes[0])
sns.histplot(np.log1p(sample), bins=60, ax=axes[1])
axes[0].set(title=f"판매량 분포 · 0 판매 {zero_rate:.1%}", xlabel="일별 판매량 (매장 × 상품군)", ylabel="기록 수 (행)")
axes[1].set(title="로그 변환 후 판매량", xlabel="log(1 + 일별 판매량)", ylabel="기록 수 (행)")
fig.tight_layout()
plt.show()

### **이 분포가 의사결정에 주는 정보**

| 관찰 | 모델·분석에서의 판단 | 운영에서의 의미 |
|---|---|---|
| 0 판매 기록이 31.3% | 0·소량 판매 구간의 오류를 별도로 확인 | 낮은 판매량의 예측을 전체 평균 성능만으로 판단하지 않음 |
| 일부 기록의 판매량이 큼 | MAE·WAPE와 RMSLE를 함께 보고 수요 규모별 오류를 분석 | 큰 수요의 오차가 주간 물량 계획에 미치는 영향을 점검 |

이 그림만으로 개별 매장·상품군의 발주량을 정할 수는 없습니다. 0 판매도 재고 부족이나 판매 중단 때문일 수 있어 **수요 0으로 단정할 수 없습니다**. 실제 활용 가치는 이후의 동일 조건 기준선 비교와 매장·상품군별 오류 분석에서 확인합니다.


## **전체 판매 추이와 요일별 패턴**

- 일별 전체 판매량과 28일 평균으로 **공급 계획이 감당해야 하는 규모 변화**를 봅니다.
- 요일별 반복이 보이면 다음 주 같은 요일의 판매량을 단순 기준선으로 사용할 근거가 됩니다. 전체 합계의 패턴이 모든 매장·상품군에 똑같이 적용되지는 않습니다.

In [ ]:
daily = daily_sales(train)
daily["rolling_28d"] = daily["sales"].rolling(28, min_periods=1).mean()
daily["dayofweek"] = daily["date"].dt.day_name().str[:3]
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=daily, x="date", y="sales", alpha=0.25, linewidth=0.7, ax=axes[0])
sns.lineplot(data=daily, x="date", y="rolling_28d", linewidth=1.5, ax=axes[0])
sns.boxplot(data=daily, x="dayofweek", y="sales", order=weekday_order,
            showfliers=False, ax=axes[1])
axes[0].set(title="일별 판매량과 28일 평균", xlabel="", ylabel="판매량")
axes[1].set(title="요일별 판매량", xlabel="", ylabel="판매량")
fig.tight_layout()
plt.show()

## **주간 주기성 확인**

- 1일과 7일 간격 및 7일의 배수에서 자기상관을 비교합니다. 값이 클수록 과거 해당 날짜의 전체 판매량과 함께 움직였다는 뜻입니다.
- 전체 추세가 상관을 키우는지 확인하려고 28일 중심 평균을 뺀 값도 봅니다. **중심 평균은 EDA에만 쓰고 예측 Feature에는 쓰지 않습니다.**

In [ ]:
seasonal_lags = [1, 7, 14, 21, 28]
daily["trend_28d"] = daily["sales"].rolling(28, center=True, min_periods=14).mean()
daily["detrended_sales"] = daily["sales"] - daily["trend_28d"]
lag_correlation = pd.DataFrame({
    "lag": seasonal_lags,
    "raw_sales": [daily["sales"].autocorr(lag) for lag in seasonal_lags],
    "detrended_sales": [daily["detrended_sales"].autocorr(lag) for lag in seasonal_lags],
})
lag_overview = lag_correlation.set_index("lag").round(3)
lag_overview


### **주간 반복 패턴의 의미**

- 전날(lag 1)보다 같은 요일인 7일 전(lag 7)의 원래 판매량과 더 비슷합니다.
- 주변 28일 평균을 빼도 7일 간격의 관계가 남아, 전체 판매 추세만으로 설명되지 않는 주간 패턴을 확인할 수 있습니다.
- 여기서 lag는 **오늘을 기준으로 며칠 전인지**를 뜻합니다.

| 비교 시점 | 원래 판매량과의 상관 | 추세 제거 후 상관 | 해석 |
|---|---:|---:|---|
| 전날 (lag 1) | 0.767 | 0.371 | 가까운 날짜의 판매가 비슷하지만, 추세를 빼면 관계가 약해집니다. |
| 전주 같은 요일 (lag 7) | **0.826** | 0.531 | 주간 반복 패턴이 뚜렷합니다. |
| 2주 전 같은 요일 (lag 14) | 0.822 | **0.608** | 추세를 빼도 강한 관계가 남습니다. |
| 3~4주 전 같은 요일 | 약 0.77 | 약 0.55~0.60 | 주간 패턴이 몇 주간 유지됩니다. |




In [ ]:
lag_plot = lag_correlation.melt(
    id_vars="lag", value_vars=["raw_sales", "detrended_sales"],
    var_name="계산 방식", value_name="상관계수",
)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=lag_plot, x="lag", y="상관계수", hue="계산 방식", ax=ax)
ax.set(title="과거 며칠 전 판매와 오늘 판매의 상관", xlabel="간격 (일)")
fig.tight_layout()
plt.show()

같은 요일의 과거 판매량이 전날 판매량보다 강하게 연결됩니다. 이를 근거로 전주 동일 요일과 최근 4주 동일 요일 평균을 기준선으로 삼고, 모델에도 7·14·28일 lag를 포함합니다. 실제 예측 성능은 시계열 Test 결과로 판단합니다.


## **상품군과 매장의 판매 규모**

- 판매 수량이 집중된 상품군과 매장을 확인합니다. 본사는 이 집계로 **어느 영역부터 수요 계획을 검토할지** 정할 수 있습니다.
- 판매 수량 비중은 재고 부족 위험이나 이익 비중과 같지 않습니다. 그런 판단에는 재고·가격·비용 자료가 더 필요합니다.

In [ ]:
family_sales = (
    train.groupby("family", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)
store_sales = (
    train.groupby("store_nbr", observed=True)["sales"].sum()
    .sort_values(ascending=False).head(15).sort_values()
)

family_share_pct = 100 * family_sales / train["sales"].sum()
store_share_pct = 100 * store_sales / train["sales"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(x=family_share_pct.values, y=family_share_pct.index, ax=axes[0], color="steelblue")
sns.barplot(x=store_share_pct.values, y=store_share_pct.index.astype(str), ax=axes[1], color="steelblue")
axes[0].set(title="판매 수량 비중 상위 상품군", xlabel="전체 판매 수량 중 비중 (%)", ylabel="")
axes[1].set(title="판매 수량 비중 상위 매장", xlabel="전체 판매 수량 중 비중 (%)", ylabel="매장")
fig.tight_layout()
plt.show()

## **프로모션과 판매량**

아래 차이는 인과효과가 아니라 단순 집단 비교입니다. 상품군과 시점이 다르므로 모델 변수의 필요성을 판단하는 탐색 결과로만 사용합니다.

In [ ]:
promotion_overview = promotion_summary(train)
promotion_overview


In [ ]:
promotion_by_family = (
    train.assign(promoted=train["onpromotion"].gt(0))
    .groupby(["family", "promoted"], observed=True)["sales"].mean()
    .unstack()
    .dropna()
)
promotion_by_family["mean_difference"] = promotion_by_family[True] - promotion_by_family[False]
promotion_family_view = promotion_by_family.sort_values("mean_difference", ascending=False).head(15)
promotion_family_view


## **상품군별 0 판매 비율**

- 0 판매가 잦은 상품군은 작은 예측 오차에도 비율 지표가 크게 흔들릴 수 있습니다.
- 이 결과를 02의 수요 유형, 08의 유형별 **판매량 비중·7일 오류**와 연결해 봅니다.

In [ ]:
zero_rate_by_family = (
    train.assign(zero_sales=train["sales"].eq(0))
    .groupby("family", observed=True)["zero_sales"]
    .mean().sort_values(ascending=False)
)
top_zero = zero_rate_by_family.head(15).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=top_zero.values, y=top_zero.index, ax=ax, color="steelblue")
ax.set(title="0 판매 비율이 높은 상품군", xlabel="0 판매 비율", ylabel="")
fig.tight_layout()
plt.show()

## **국가 공휴일과 전체 판매량**

전국 단위 공휴일의 단순 비교이며 공휴일의 인과효과로 해석하지 않습니다.

In [ ]:
national_holiday_dates = holidays.loc[
    holidays["locale"].eq("National")
    & ~holidays["transferred"]
    & ~holidays["type"].eq("Work Day"),
    "date",
].unique()
holiday_sales = daily.assign(
    national_holiday=daily["date"].isin(national_holiday_dates)
).groupby("national_holiday")["sales"].agg(days="size", mean="mean", median="median")
holiday_sales

## **1차 확인 사항**

- 0 판매와 판매 규모 차이를 함께 보기 위해 RMSLE·MAE·WAPE를 나누어 해석합니다. 발주 목적에서는 7일 합계 WAPE와 Bias도 확인합니다.
- 요일과 장기 추세가 보여 달력 변수와 과거 판매량 변수가 필요합니다.
- 프로모션 집단의 차이는 크지만 인과관계로 해석하지 않습니다.
- 상품군과 점포 간 판매 규모가 달라 범주형 변수를 유지합니다.
- 0 판매가 많은 상품군과 국가 공휴일은 오류 분석에서 별도 구간으로 확인합니다.

## **EDA 핵심 결과 정리**

아래 숫자는 앞에서 계산한 전체 이력의 기술 통계를 한눈에 모은 것입니다. 0 판매 비율은 희소 수요를 따로 볼 필요가 있음을, 추세를 제거한 7일 자기상관은 동일 요일 기준선을 시험할 근거를 보여줍니다. 상위 상품군의 판매 비중은 전체 지표가 큰 상품군에 좌우될 수 있음을 설명합니다.

**전체 기간 EDA는 설명 자료**입니다. 군집 소속과 모델별 Test 성능을 연결할 때는 [02 수요 패턴 군집](02_demand_segments.ipynb)처럼 Valid 시작 전까지만 사용합니다. 판촉일과 판매량의 관계도 인과효과로 해석하지 않습니다.

In [ ]:
top_family = family_sales.index[-1]
top_family_share = family_sales.iloc[-1] / daily["sales"].sum()
weekly_correlation = lag_correlation.loc[
    lag_correlation["lag"].eq(7), "detrended_sales"
].iloc[0]

eda_summary = pd.DataFrame([
    {"관찰": "0 판매 비율", "값": f"{zero_rate:.1%}", "연결": "희소 수요의 오류를 별도 분석"},
    {"관찰": "추세 제거 후 7일 자기상관", "값": f"{weekly_correlation:.3f}",
     "연결": "동일 요일 기준선과 요일성 Feature"},
    {"관찰": f"최대 판매 상품군 ({top_family}) 비중", "값": f"{top_family_share:.1%}",
     "연결": "전체 WAPE와 상품군별 WAPE를 함께 보고"},
])
eda_summary